# CodSoft Data Analytics Internship — Task 1
## Data Cleaning & Preprocessing

**Goal:** Inspect a messy customer dataset, identify missing values and duplicates, standardize inconsistent values, correct data types, and save a cleaned CSV.

This notebook uses a small practice dataset included in the `data` folder. Run the cells from top to bottom.

## 1. Import libraries and load the dataset
Pandas helps us read, inspect, and clean tabular data.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()
# If your notebook is opened from another folder, change PROJECT_DIR to the CODSOFT_TASK1 folder.
INPUT_FILE = PROJECT_DIR / "data" / "customers_raw.csv"
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_FILE = OUTPUT_DIR / "customers_cleaned.csv"

df = pd.read_csv(INPUT_FILE)
print("Dataset loaded successfully!")
df.head()

## 2. Inspect the dataset
`shape` gives rows and columns; `info()` shows data types and non-null counts; `describe()` summarizes numeric columns.

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn information:")
df.info()
print("\nNumeric summary:")
display(df.describe())

## 3. Find missing values and duplicates
We inspect these before changing the data so we understand the quality issues.

In [ ]:
print("Missing values in each column:")
display(df.isnull().sum().to_frame("Missing count"))

print("Fully duplicated rows:", df.duplicated().sum())
print("\nUnique values in text columns:")
for col in ["Gender", "City"]:
    print(f"{col}:", sorted(df[col].dropna().astype(str).unique()))

## 4. Remove duplicate rows and standardize text
Extra spaces and differences such as `PUNE`, `Pune`, and `pune` can make categories inconsistent.

In [ ]:
df = df.drop_duplicates().copy()

for col in ["Name", "City", "Gender"]:
    df[col] = df[col].astype("string").str.strip()

df["Name"] = df["Name"].str.title()
df["City"] = df["City"].str.title()
df["Gender"] = df["Gender"].str.lower().replace({"f": "female", "m": "male"}).str.title()
df["Gender"] = df["Gender"].replace({"Unknown": pd.NA, "": pd.NA})

display(df.head())

## 5. Correct data types and flag invalid values
Unparseable dates become missing. Values outside reasonable ranges are treated as invalid rather than kept as facts.

In [ ]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["PurchaseAmount"] = pd.to_numeric(df["PurchaseAmount"], errors="coerce")
df["Satisfaction"] = pd.to_numeric(df["Satisfaction"], errors="coerce")
df["JoinDate"] = pd.to_datetime(df["JoinDate"], errors="coerce", dayfirst=True)

df.loc[~df["Age"].between(0, 100), "Age"] = pd.NA
df.loc[df["PurchaseAmount"] < 0, "PurchaseAmount"] = pd.NA
df.loc[~df["Satisfaction"].between(1, 5), "Satisfaction"] = pd.NA

print(df.dtypes)
display(df.isnull().sum().to_frame("Missing after validation"))

## 6. Handle missing values
Use the median for numeric fields and the most frequent category for city/gender. Invalid dates remain missing because a date should not be guessed.

In [ ]:
for col in ["Age", "PurchaseAmount", "Satisfaction"]:
    df[col] = df[col].fillna(df[col].median())

for col in ["Gender", "City"]:
    mode_value = df[col].mode(dropna=True)
    if not mode_value.empty:
        df[col] = df[col].fillna(mode_value.iloc[0])

df["Age"] = df["Age"].round().astype("Int64")
df["Satisfaction"] = df["Satisfaction"].round().astype("Int64")
df["PurchaseAmount"] = df["PurchaseAmount"].round(2)

display(df.head())
print("Missing values after cleaning:")
display(df.isnull().sum().to_frame("Missing count"))

## 7. Export the cleaned CSV
This completes the bonus task in the internship instructions.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False, date_format="%Y-%m-%d")
print("Cleaned file saved to:", OUTPUT_FILE.resolve())
print("Final dataset shape:", df.shape)
print("Duplicate rows remaining:", df.duplicated().sum())

## 8. Conclusion

In this project, we:
- loaded and inspected a CSV using Pandas;
- identified missing values and duplicate records;
- standardized inconsistent text entries;
- converted columns to suitable data types;
- handled invalid values and missing data using documented rules; and
- exported a cleaned dataset to `output/customers_cleaned.csv`.

**Before submitting:** run every cell, check the generated CSV, and practice explaining why each cleaning step was used.